# AAT6153 Chapter 2 — Demo
## Pre-trained Word Embeddings with GloVe
**Slides:** Word Embeddings (19–21), Pre-trained Word Embeddings (25–29), Visualising Word Embeddings (30)

In this notebook you will:
1. Download the **GloVe** vectors (Stanford, glove.6B, 100 dimensions) and parse them (slide 27).
2. Check that similar words are close together, unlike one-hot vectors (slides 18–19).
3. Try the famous analogy **king − man + woman ≈ queen** (slide 20).
4. Find **nearest neighbours** and plot words in 2-D with **PCA** (slide 30).
5. Load GloVe into a **frozen** Keras `Embedding` layer (slide 28).

*Runtime: CPU is fine. The download is 822 MB and takes a few minutes.*

---
## Download GloVe
The cell tries several mirrors of the same `glove.6B.zip` file, in turn, until one works, and checks the download before extracting the 100-dimensional vectors. If the file is already there (e.g. you re-run the notebook), it skips the download.

In [ ]:
import os, zipfile

GLOVE_ZIP, GLOVE_TXT = "glove.6B.zip", "glove.6B.100d.txt"
MIRRORS = [
    "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip",  # Stanford NLP's Hugging Face account
    "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip",             # Stanford
    "https://nlp.stanford.edu/data/glove.6B.zip",                          # Stanford (original link)
]

have_zip = os.path.exists(GLOVE_ZIP) and zipfile.is_zipfile(GLOVE_ZIP)   # e.g. uploaded by hand
if not os.path.exists(GLOVE_TXT) and not have_zip:
    for url in MIRRORS:
        print("Downloading from", url)
        !wget -q --show-progress --tries=3 --timeout=60 -O {GLOVE_ZIP} {url}
        if os.path.exists(GLOVE_ZIP) and zipfile.is_zipfile(GLOVE_ZIP):
            break
        print("  Download failed, trying the next mirror...")
        if os.path.exists(GLOVE_ZIP):
            os.remove(GLOVE_ZIP)
    else:
        raise RuntimeError("Could not download GloVe from any mirror. Download glove.6B.zip in your browser "
                           "from https://nlp.stanford.edu/projects/glove/ and upload it to Colab (Files panel), "
                           "then re-run this cell.")

if not os.path.exists(GLOVE_TXT):
    with zipfile.ZipFile(GLOVE_ZIP) as z:
        z.extract(GLOVE_TXT)                 # only the 100-d file (about 330 MB)
print(f"{GLOVE_TXT}: {os.path.getsize(GLOVE_TXT) / 1e6:.0f} MB")

## Step 1: Parse the GloVe file into a dictionary (slide 27)

In [ ]:
import numpy as np
import keras
from keras import layers

embeddings_index = {}
with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        embeddings_index[word] = np.asarray(coefs.split(), dtype="float32")
print(f"Found {len(embeddings_index)} word vectors.")
print("Vector for 'cat' (first 10 of 100 values):", embeddings_index["cat"][:10].round(3))

---
## Similar words are close together (slides 18–19)
Compare with the one-hot demo, where every cosine similarity was 0.

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

e = embeddings_index
for w1, w2 in [("cat", "dog"), ("cat", "car"), ("car", "truck"), ("happy", "sad"), ("happy", "table")]:
    print(f"cosine({w1:5s}, {w2:5s}) = {cosine(e[w1], e[w2]):.3f}")

## Nearest neighbours (slide 30 exercise)

In [ ]:
words = list(embeddings_index.keys())[:50000]            # the 50,000 most frequent words
matrix = np.stack([embeddings_index[w] for w in words])
matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)

def nearest(vec, k=8, exclude=()):
    sims = matrix_norm @ (vec / np.linalg.norm(vec))
    best = [i for i in np.argsort(sims)[::-1] if words[i] not in exclude][:k]
    return [(words[i], round(float(sims[i]), 3)) for i in best]

for w in ["computer", "malaysia", "happy"]:
    print(f"{w:9s} →", nearest(e[w], exclude={w}))

## Word analogies (slide 20)
**king − man + woman ≈ ?**

In [ ]:
def analogy(a, b, c, k=5):
    """a is to b as c is to ?   (b - a + c)"""
    return nearest(e[b] - e[a] + e[c], k=k, exclude={a, b, c})

print("man : king  :: woman :", analogy("man", "king", "woman"))
print("france : paris :: malaysia :", analogy("france", "paris", "malaysia"))
print("walk : walked :: swim :", analogy("walk", "walked", "swim"))

## Visualising embeddings in 2-D with PCA (slide 30)
PCA keeps the two directions with the most variance. Distances in the plot are only approximate.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

plot_words = ["king", "queen", "man", "woman", "prince", "princess",
              "cat", "dog", "horse", "cow",
              "car", "truck", "bus", "train",
              "happy", "sad", "angry", "joyful"]
pts = PCA(n_components=2).fit_transform(np.stack([e[w] for w in plot_words]))

plt.figure(figsize=(9, 7))
plt.scatter(pts[:, 0], pts[:, 1])
for w, (px, py) in zip(plot_words, pts):
    plt.annotate(w, (px, py), xytext=(4, 4), textcoords="offset points", fontsize=11)
plt.title("GloVe vectors projected to 2-D with PCA")
plt.show()

Explore further in the **TensorFlow Embedding Projector**: https://projector.tensorflow.org (load *Word2Vec All* and search for a word).

---
## Step 2: Build a frozen Embedding layer from GloVe (slide 28)
Row *i* of the embedding matrix is the GloVe vector of the word with index *i* in **our** vocabulary. Words missing from GloVe keep a zero vector.

In [ ]:
corpus = ["the movie was great", "a wonderful film", "bad and awful acting",
          "the plot was boring but the acting was superb"]
vectorizer = layers.TextVectorization(max_tokens=20000, output_sequence_length=200)
vectorizer.adapt(corpus)

vocab = vectorizer.get_vocabulary()       # from a fitted TextVectorization
embedding_dim = 100
embedding_matrix = np.zeros((len(vocab), embedding_dim))
for i, word in enumerate(vocab):
    vector = embeddings_index.get(word)
    if vector is not None:                # unknown words stay all-zeros
        embedding_matrix[i] = vector

embedding_layer = layers.Embedding(len(vocab), embedding_dim,
                                   trainable=False)   # freeze GloVe
embedding_layer.build((1,))
embedding_layer.set_weights([embedding_matrix])

found = int((np.abs(embedding_matrix).sum(axis=1) > 0).sum())
print(f"Vocabulary size: {len(vocab)},  found in GloVe: {found}")
print("Output shape for a batch of 1 review:", embedding_layer(vectorizer(["a great film"])).shape)
print("Trainable parameters:", len(embedding_layer.trainable_weights))

---
## Static vs contextual embeddings (slide 29)
GloVe gives **one** vector per word, so *bank* is the same in "river bank" and "bank loan". Transformer models (BERT, GPT) give context-dependent vectors, which you will meet in later chapters.

## Try it yourself
1. Find the nearest neighbours of **"bank"**. Do they mix the river and money senses?
2. Try your own analogies, e.g. *malaysia : ringgit :: japan : ?*
3. Repeat with `glove.6B.300d.txt` (unzip it from the same file). Do the analogies improve?

In [ ]:
# Use this cell for your experiments